# Project 1

Welcome to the first NLP1 project. The goal of the first tutorial was to get you familiar with text segmentation and simple text classification for topic detection.
Given the skills you have developed in the first week, you will now be applying them within a different dataset and task.

We are recommending a set of tasks and associated datasets; using those, you will be experimenting on problems introduced during the tutorials. You will be applying hands-on the experiments you have already explored in the last tutorial, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and analysing errors. 

The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

As the tutorials will be building upon your knowledge throughout the weeks, the task and dataset you choose now will be the ones you use in projects 2 and 3; and the ones you will be finally writing about in your final blog post.  

## Get to work!

The next section has a checklist to help you not lose track of what needs to be done. Here we have a more informal guide, with prompts and questions for you. 


1. [Choose a dataset](http://nlp-illc.github.io/nlp1/). Spend some time understanding how the dataset looks and some statistics of it (e.g. how many instances of each classification label we have?)
2. Remember what we learnt during the tutorial: to develop any classifier of the kind we saw, we needed a way to segment text. Do you want to use the tokeniser we have developed? Do you need to train a new one? If you do, what hyperparameters shall you use? What data shall you use to train it? Consider that your dataset may be too small to train a tokeniser! Think whether you will need a larger resource of similar nature to train a tokeniser.
3. Start thinking how to represent your documents into features. We have experimented with token occurences and frequencies. Can you think of any other ways, possibly? 
4. We had experimented with Logistic Regression and a shallow neural network. You can experiment with and assess different architectures now and their hyper parameters (e.g. add layers, possibly change the activation functions, drop out rates).
5. Evaluate your models! Was the evaluation method adequate for your new experimental setup? Do you want to assess other aspects of performance?
6. Analyse the various models you assessed and try to decide on which is the best.

Be mindful that you will be using your experiments to write up your final blog, so make sure they are clean and neat!

You can write your own code and you can reuse code from the tutorial and its packages or from other sources. You may use coding assistants if you like, but use it mindfully as you take full responsibility for this kind of (semi-)automated coding. Also, keep a record of the kinds of things you delegate to any form of assistant, as you will be asked to disclose this in your blog post (not because we will pass a value judgement on it, but because it is a widespread and good practice to document such things).

You may use jupyter notebooks or python files, it's up to you as we will not directly assess your code. But, once again, be organised, as we will ask you to share a reproducible version of your experiments along with your blog post (again, because this is how real research is publicised). 

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

## Checklist

What **we expect you to do**:

- [ ] Compare different systems of text classification for your choice of data/task.
- Your comparisons should cover:
    - [ ] the two representation spaces in the tutorial (token indicators and bag-of-words) and an additional one of your choice.
    - [ ] the two kinds of architectures in the tutorial (log-linear and single-hidden-layer MLP) and an additional one of your choice (such as a modification of the MLP).
- [ ] Your comparisons must be based on cross-validation and a relevant metric (accuracy or else).
- [ ] You should make use of Bayesian estimation to help you analyse results in relation to a region of practical equivalence (ROPE), and you can motivate the ROPE size yourself.

What you need to **bring to the oral assessment**:
* You will be asked questions about the points above, you can bring a slide deck with up to five slides (one for each of the following topics _data and metrics_, _features_, _architectures_, _results_, _extra_); you _will not_ give a presentation, rather, these slides are to be thought of as "backup slides" that you can use to guide your answers to the questions the TA will ask you.
* You may also be asked questions about the tutorial, for those, you don't need to prepare any deliverable (a good way to prepare is to work through the tutorial and reflect on the questions there).

What you should **keep in mind** given that you will eventually write a blog post about your work throughout the course:
* Organise your code;
* Summarise your motivations and findings;
* Document your sources and any form of AI-assistance you use.

In [31]:
%load_ext autoreload
%autoreload 2    

In [101]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) 
import sys
from pathlib import Path
import pandas as pd
import part1
from part1.clslib import NamedClasses
from part1.experiment import train_tokenizer
from part1.clslib import TokenIndicator, BagOfWords


In [102]:
# Assuming the working directory is root/part1 or root/part2
root = Path.cwd().parent
sys.path.insert(0, str(root))


# save models and outputs there to keep the directory tree organised
results_folder = Path("./results")
results_folder.mkdir(parents=True, exist_ok=True) 
results_folder

WindowsPath('results')

In [64]:
# sentiment classification dataset from stanfordnlp
splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
base = "hf://datasets/stanfordnlp/sst2/"
df_train = pd.read_parquet(base + splits["train"])
df_validation = pd.read_parquet(base + splits["validation"])
df_test = pd.read_parquet(base + splits["test"])


In [65]:
print('Training data:')
df_train.info()
print()

print('Validation data:')
df_validation.info()
print()

print('Test data:')
df_test.info()

Training data:
<class 'pandas.DataFrame'>
RangeIndex: 67349 entries, 0 to 67348
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   idx       67349 non-null  int32
 1   sentence  67349 non-null  str  
 2   label     67349 non-null  int64
dtypes: int32(1), int64(1), str(1)
memory usage: 4.7 MB

Validation data:
<class 'pandas.DataFrame'>
RangeIndex: 872 entries, 0 to 871
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   idx       872 non-null    int32
 1   sentence  872 non-null    str  
 2   label     872 non-null    int64
dtypes: int32(1), int64(1), str(1)
memory usage: 107.3 KB

Test data:
<class 'pandas.DataFrame'>
RangeIndex: 1821 entries, 0 to 1820
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   idx       1821 non-null   int32
 1   sentence  1821 non-null   str  
 2   label     1821 non-null   int64
d

In [ ]:
outcome_space = NamedClasses(['negative', 'positive'])
pairs = list(zip(
    df_train["sentence"],
    df_train["label"].astype(int)
))
print(outcome_space)
print(pairs[2])

0: negative, 1: positive
('that loves its characters and communicates something rather beautiful about human nature ', 1)


In [ ]:
# Train new bpe tokenizer
corpus = df_train["sentence"].astype(str)

tokenizer = train_tokenizer(
    corpus=corpus,
    vocab_size=5000,
    min_frequency=10
)
print(corpus)

tokenizer_path = results_folder / "sst2_tokenizer.json"
tokenizer.save(str(tokenizer_path))

0             hide new secretions from the parental units 
1                     contains no wit , only labored gags 
2        that loves its characters and communicates som...
3        remains utterly satisfied to remain the same t...
4        on the worst revenge-of-the-nerds clichés the ...
                               ...                        
67344                                 a delightful comedy 
67345                     anguish , anger and frustration 
67346    at achieving the modest , crowd-pleasing goals...
67347                                    a patient viewer 
67348    this new jangle of noise , mayhem and stupidit...
Name: sentence, Length: 67349, dtype: str


In [ ]:

# 1. Token indicator
indicator_encoder = TokenIndicator(tokenizer)

# 2. Bag Of Words
bow_encoder = BagOfWords(tokenizer)

# 3. TF-IDF

In [ ]:
# Log-linear model

In [ ]:
# MLP with 1 hidden layer

In [ ]:
# Decision tree or something else